# Experimentando com bancos de dados em Python

## Conectando com a base de dados

Existem sistemas de gerenciamento de bancos de dados que operam no mesmo processo da aplicação cliente - ou seja, são bibliotecas que você pode importar diretamente no seu programa. Por exemplo: a biblioteca ``sqlite3`` (https://docs.python.org/3/library/sqlite3.html) é parte do conjunto de bibliotecas padrão da linguagem Python, e permite criar bases de dados diretamente em arquivo ou memória, sem a necessidade de um programa em separado. Eis um exemplo de uso de sqlite3:

In [1]:
import sqlite3

db = sqlite3.connect("teste.db")
# Para criar a base em memória (base temporária) use o nome de arquivo
# ":memory:"

Note que o arquivo ``teste.db`` foi criado. 

In [2]:
import os

print("teste.db" in os.listdir())

True


Apesar do ``sqlite3`` criar o sistema de gerenciamento de banco de dados (SGBD) SQLite diretamente no nosso processo (neste caso, o kernel Python que está sendo executado por esse notebook), o objeto de acesso ainda é uma "conexão". Isso se deve ao fato de que SGBDs "normais" - aqueles que realmente usamos em produção - são geralmente *sistemas cliente-servidor*, onde o SGBD é um processo separado da aplicação cliente e a comunicação entre eles acontece através de uma conexão de rede.

<center><img src='imgs/client-server.png'></center>

**Atividade:**

Na figura acima, quem é o cliente do sistema de gerenciamento de banco de dados?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sua resposta aqui! Dê dois cliques e edite.
    
    Aplicação Python

</div>

Uma conexão é simplesmente o conduíte entre o cliente e o SGBD. Em Python, a especificação PEP 249 (https://www.python.org/dev/peps/pep-0249/) define que uma conexão deve ter os métodos ``.close()`` (para fechar a conexão), ``.cursor()`` (para obter um *cursor*, que é o objeto para realmente interagir com o banco de dados), e o par ``.commit()``/``.rollback()`` (para gerenciamento de transações).

Vamos fechar essa conexão e trabalhar com o MySQL daqui em diante.

In [3]:
db.close()

In [4]:
os.remove("teste.db")

Antes de tudo vamos criar um banco de dados. Queremos criar um banco de dados para gerenciar as equipes do primeiro campeonato Insper de ninja-cowboy-urso!

- Ninja, com sua velocidade e furtividade, detona o cowboy.
- O cowboy atira no urso.
- O urso dá uma patada no ninja e acaba com ele.

Cada jogador tem um nome, uma equipe, e sua jogada favorita. As equipes tem um nome e um grito de guerra.

Portanto, nosso schema será:

```
jogador (id PK, nome_equipe, nome, preferencia)
equipe (nome PK, grito)
```

com a restrição:

```
jogador(nome_equipe) FK para equipe(nome)
```

E um dicionário de dados simples para este problema é:

<center><b>jogador</b></center>
<table>
    <tr>
        <th>Campo</th>
        <th>Tipo</th>
        <th>Significado</th>
        <th>Chave?</th>
        <th>Valores</th>
    </tr>
    <tr>
        <td>id</td>
        <td>int</td>
        <td>Identificador unico do jogador</td>
        <td>PK</td>
        <td>sem restrições</td>
    </tr>
    <tr>
        <td>nome_equipe</td>
        <td>string</td>
        <td>Nome da equipe na qual o jogador atua</td>
        <td>FK equipe(nome)</td>
        <td></td>
    </tr>
    <tr>
        <td>nome</td>
        <td>string</td>
        <td>Nome do jogador</td>
        <td></td>
        <td>Pode haver dois jogadores com mesmo nome. Máximo 80 caracteres.</td>
    </tr>
    <tr>
        <td>preferencia</td>
        <td>int</td>
        <td>Jogada preferida do jogador</td>
        <td></td>
        <td>0: Ninja, 1: Cowboy, 2: Urso</td>
    </tr>
</table>

<center><b>equipe</b></center>
<table>
    <tr>
        <th>Campo</th>
        <th>Tipo</th>
        <th>Significado</th>
        <th>Chave?</th>
        <th>Valores</th>
    </tr>
    <tr>
        <td>nome</td>
        <td>string</td>
        <td>Nome da equipe</td>
        <td>PK</td>
        <td>Máximo 30 caracteres</td>
    </tr>
    <tr>
        <td>grito</td>
        <td>string</td>
        <td>Grito de guerra da equipe</td>
        <td></td>
        <td>Maximo 80 caracteres</td>
    </tr>
</table>

**Atividade:** Escreva e rode o script de criação do banco de dados. Chame o banco de dados de `torneio`. Compare com o **gabarito** em `sql/script_001.sql`.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Execute, no **Workbench** o script para criação da base `torneio` disponível em `sql/script_001.sql`.

</div>

Para conectar com o MySQL precisamos da biblioteca *mysql-connector-python*.
 
Para instalar esta biblioteca (já está nos `requirements.txt`, provavelmente não precisará instalar):

**`!python -m pip install mysql-connector-python==8.3.0`**

In [5]:
from dotenv import load_dotenv
import insperautograder.jupyter as ia
import mysql.connector
import os

In [6]:
print(f"mysql.connector.__version__: {mysql.connector.__version__}")

mysql.connector.__version__: 8.3.0


In [7]:
load_dotenv(override=True)

True

Vamos agora abrir uma conexão com o MySQL. Como agora temos realmente um sistema cliente-servidor, precisamos definir onde está o servidor, e precisamos também de informações de autenticação.

In [8]:
connection_options = {
    "host": "localhost",
    "user": "root",
    "password": "123456",
    "database": "torneio"
}

**Pergunta**: Faz sentido guardar o *user* e *password* dentro do próprio código fonte? Qual o jeito correto de lidar com username e password no seu produto?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sua resposta aqui! Dê dois cliques e edite.
    
    coloca no .env

</div>

**Resposta**: 

Não se deve guardar *user* e *password* dentro do código fonte, pois trata-se de:
- Informação sensível e que não deve ser acessada pelos programadores.
- Informação que muda com frequência - ou pelo menos deveria mudar!
- Dado puro, que deve ser separado do código.

Algumas alternativas populares para resolver esse problema são:
- Use um arquivo de configuração na conta que será usada para rodar a aplicação. Este arquivo deve ter todas as permissões de acesso negadas, exceto permissão de leitura para o próprio usuário da conta de execução da aplicação. O arquivo deverá ser lido então pela aplicação no momento da inicialização.
- Use uma variável de ambiente com essas credenciais. É a mesma coisa que no ítem anterior, só ao invés de ter um arquivo de configuração da aplicação, temos o arquivo de configuração da conta em si, com os mesmos requisitos em relação às restrições de permissão.

## Atualizando `.env`

### Tarefa:

Atualize o arquivo `.env`. Por enquanto ele provavelmente contém apenas as variáveis criadas para a API de correção automática:

```python
IAG_SERVER_URL="https://bigdata.insper-comp.com.br/iag"
IAG_SUBJECT="megadados"
IAG_OFFERING="xxxx"
IAG_TOKEN="iagtok_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
```

Vamos adicionar três novas variáveis:

- `MD_DB_SERVER`: o endereço para o host onde o mysql está instalado (provavelmente `"localhost"`)
- `MD_DB_USERNAME`: o usuário para acesso ao banco (provavelmente `"root"`)
- `MD_DB_PASSWORD`: a senha do usuário.


O `.env` ficará neste formato (veja as três novas variáveis ao final):

```python
IAG_SERVER_URL="https://bigdata.xxxxxxx"
IAG_SUBJECT="alguma_disciplina"
IAG_OFFERING="xxx"
IAG_TOKEN="iagtok_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
MD_DB_SERVER="localhost"
MD_DB_USERNAME="root"
MD_DB_PASSWORD="alguma_senha"
```

In [14]:
load_dotenv(override=True)

connection_options = {
    "host": os.getenv("MD_DB_SERVER"),
    "user": os.getenv("MD_DB_USERNAME"),
    "password": os.getenv("MD_DB_PASSWORD"),
    "database": "torneio"
}

**Abrindo a conexão:**

In [18]:
# A notação de duplo asterisco em Python permite usar um dicionário 
# no lugar de argumentos nomeados. A linha a ser executada abaixo equivale á:
#
# connection = mysql.connector.connect(
#     host=os.getenv("MD_DB_SERVER"),
#     user=os.getenv("MD_DB_USERNAME"),
#     password=os.getenv("MD_DB_PASSWORD"),
#     database="torneio"
# )
# )
#
connection = mysql.connector.connect(**connection_options)

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

**IMPORTANTE:**

TODAS AS OPERAÇÕES REALIZADAS NESTA CONEXÃO SÃO TEMPORÁRIAS, ATÉ QUE O COMANDO `connection.commit()` seja executado!

</div>

<div class="alert alert-warning" role="alert">
    
**MAIS IMPORTANTE AINDA:**

ISSO NÃO VALE SE **AUTOCOMMIT** ESTIVER LIGADO!

</div>


Estudaremos melhor essa questão quando discutirmos transações em maiores detalhes.

Vamos criar alguns dados iniciais:

In [48]:
T1 = "Raposas Nerds"
T2 = "É nois"

NINJA = 0
COWBOY = 1
URSO = 2

times = {
    T1: "sudo vencer!",
    T2: "Olha eu mamãe!",
}

jogadores = [
    ("Raul Ayres", T1, NINJA),
    ("Luciano Hashimoto", T1, COWBOY),
    ("Rafael Montaigner", T1, URSO),
    ("Igor Miranda", T2, URSO),
    ("Andrew Ikeda", T2, COWBOY),
    ("Fábio Kurauchi", T2, NINJA),
]

Vamos inserir os dados dos times.

**Atividade:** Quem devemos inserir primeiro: dados na tabela `equipe` ou na tabela `jogador`?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sua resposta aqui! Dê dois cliques e edite.
    
    Equipe pois jogador usa fk da tabela equipe
    
</div>

Quando uma tabela tem chave estrangeira para outra tabela dizemos que ela é filha (*child*) da outra tabela, e que a outra tabela é a pai (*parent*).

Um *cursor* é o objeto que permite executar queries SQL e interagir com os resultados obtidos.

### Tentando inserir um jogador

Vamos tentar inserir um jogador (vai falhar):

In [49]:
try:
    with connection.cursor() as cursor:
        cursor.execute("INSERT INTO jogador (nome, nome_equipe, preferencia) VALUES ('Joao', 'sete e meia', 0)")
    connection.commit()
except mysql.connector.IntegrityError as exception:
    print(f"IntegrityError: {exception}")
    connection.rollback()

IntegrityError: 1452 (23000): Cannot add or update a child row: a foreign key constraint fails (`torneio`.`jogador`, CONSTRAINT `fk_equipe` FOREIGN KEY (`nome_equipe`) REFERENCES `equipe` (`nome`))


Perceba que obtemos uma **exceção**, pois estamos tentando utilizar uma equipe inexistente na tabela de `equipe`.

Isto responde nossa pergunta anterior: precisamos inserir primeiro as equipes.

Vamos fazer isto utilizando um laço que fará todas as inserções:

In [50]:
try:
    with connection.cursor() as cursor:
        # Outra opção aqui seria o uso de cursor.executemany()
        for time, grito in times.items():
            cursor.execute("INSERT INTO equipe VALUES (%s, %s)", (time, grito))
    connection.commit()
except mysql.connector.IntegrityError as exception:
    print(f"IntegrityError: {exception}")
    connection.rollback()

Note o comando ``commit`` acima: ele serve para efetivar a operação no banco de dados. Verifique no MySQL Workbench que a operação foi bem sucedida.

**Atividade:** Perceba que desta vez não especificamos os nomes das colunas no `INSERT`. É necessário deixar claro os campos da tabela?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sua resposta aqui! Dê dois cliques e edite.
    
    Ele assume a ordem dos valores como ordem das colunas

</div>

Podemos tambem usar nossa conexão para verificar o estado da tabela ``equipe``:

In [51]:
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM equipe")
    for item in cursor:
        print(item)

('É nois', 'Olha eu mamãe!')
('Raposas Nerds', 'sudo vencer!')


Podemos usar também nosso velho conhecido `pandas`!

In [52]:
import pandas as pd
pd.read_sql_query("SELECT * FROM equipe", connection)

/tmp/ipykernel_308394/1435631091.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("SELECT * FROM equipe", connection)


,nome,grito
0,É nois,Olha eu mamãe!
1,Raposas Nerds,sudo vencer!


**Atividade:** Tente executar novamente a inserção de dados, o que acontece?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sua resposta aqui! Dê dois cliques e edite.
    
    Erro de duplicados
    
</div>

## Interlúdio: sanitização de entradas de dados

Por que usamos aqueles ``%s`` no comando a ser executado? Por que não simplesmente incluir diretamente os dados na *string* do comando SQL? Por exemplo:

```Python
cursor.execute("INSERT INTO equipe VALUES ('" + time + "', '" + grito + "')")
```

Muitas vezes o resultado é o mesmo! Contudo, e SE executássemos a query abaixo...

time = "Robert', ''); DROP DATABASE torneio; -- "
grito = ""

``` python
with connection.cursor() as cursor:
    for result in cursor.execute(
            "INSERT INTO equipe VALUES ('" + time + "', '" + grito + "')",
            multi=True,
    ):
        print(result)
    connection.commit()

with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM equipe")
    for item in cursor:
        print(item)

connection.close()
```

A célula acima está desabilitada. Mas se quiser executar passe para tipo "Code" e rode. Depois, garanta que a conexão está fechada com `connection.close()`, volte para o começo do notebook e rode tudo de novo exceto a célula mortal!

![Little Bobby Tables](imgs/exploits_of_a_mom.png)

[Exploits of a mom](https://xkcd.com/327/)


Existem jeitos de quebrar o sistema com esse tipo de ataque, onde o hacker tenta inserir código SQL mal-intencionado em formulários, na expectativa de que o programador não tenha feito a sanitização das entradas. Esse ataque chama-se ***'SQL injection'***.

Fim do interlúdio, voltamos à apresentação principal.

Vamos agora inserir os jogadores. Neste caso não precisamos inserir a chave primária, pois ela é gerada automaticamente. Consequentemente temos que especificar que estamos inserindo apenas as demais colunas - temos que nomear as colunas na qual estamos inserindo os dados.

In [53]:
try:
    with connection.cursor() as cursor:
        cursor.executemany(
            "INSERT INTO jogador (nome, nome_equipe, preferencia) VALUES (%s, %s, %s)",
            jogadores)
    connection.commit()
except Exception as e:
    print(e)
    connection.rollback()

Vamos verificar se funcionou. Para evitar ter criar os cursores a todo momento e coletar os resultados vamos construir uma classe auxiliar, só para simplificar o código:

In [54]:
class ConnectionHelper:

    def __init__(self, connection):
        self.connection = connection

    def __call__(self, query, args=None):
        with self.connection.cursor() as cursor:
            print("Executando query:")
            for result in cursor.execute(query, multi=True):
                if result.with_rows:
                    for row in result.fetchall():
                        print(row)
                else:
                    print(f"{result.rowcount} linhas afetadas.")

In [55]:
db = ConnectionHelper(connection)

Note o padrão de *'dependency injection'* aqui: ao invés de criar a conexão dentro do construtor da classe ConnectionHelper, passamos um objeto 'connection' que será armazenado dentro do ConnectionHelper. Qualquer objeto 'connection' pode ser passado, de qualquer classe, desde que obedeça a interface (implícita) de um objeto de conexão ao banco de dados. Neste nosso caso, tem que ter o método `cursor()` que retorne um objeto de interação com a base de dados.

## Explorando a estrutura da base de dados

Vamos ver quais tabelas existem na base ``torneio``:

In [56]:
db("SHOW TABLES")

Executando query:
('equipe',)
('jogador',)


Para saber qual o schema da tabela `jogador`, podemos usar o comando '`DESCRIBE`'

In [57]:
db("DESCRIBE jogador")

Executando query:
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('nome', 'varchar(80)', 'NO', '', None, '')
('nome_equipe', 'varchar(30)', 'NO', 'MUL', None, '')
('preferencia', 'int', 'YES', '', None, '')


## Consultando a base de dados

Vamos usar o comando `SELECT` para listar os conteudos da tabela 'Jogador'

In [59]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(3, 'Luciano Hashimoto', 'Raposas Nerds', 1)
(4, 'Rafael Montaigner', 'Raposas Nerds', 2)
(5, 'Igor Miranda', 'É nois', 2)
(6, 'Andrew Ikeda', 'É nois', 1)
(7, 'Fábio Kurauchi', 'É nois', 0)


O comando acima lista todos os registros da tabela `jogador`, com todas as colunas presentes:

![Seleção da tabela inteira](imgs/tudo.png)

Vamos agora selecionar apenas algumas colunas para exibir.

In [60]:
db("SELECT nome, nome_equipe FROM jogador")

Executando query:
('Raul Ayres', 'Raposas Nerds')
('Luciano Hashimoto', 'Raposas Nerds')
('Rafael Montaigner', 'Raposas Nerds')
('Igor Miranda', 'É nois')
('Andrew Ikeda', 'É nois')
('Fábio Kurauchi', 'É nois')


Agora vemos apenas as colunas escolhidas.  A operação de seleção de colunas chama-se **projeção**.

![Projeção](imgs/projecao.png)

Vamos agora atuar na escolha de linhas, selecionando quais desejamos. Para escolher todas as linhas cujo `nome_equipe` começa com 'Rap' podemos executar a query a seguir:

In [61]:
db("SELECT * FROM jogador WHERE nome_equipe LIKE 'Rap%'")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(3, 'Luciano Hashimoto', 'Raposas Nerds', 1)
(4, 'Rafael Montaigner', 'Raposas Nerds', 2)


Vemos apenas as linhas escolhidas. A operação de filtragem de linhas apropriadas chama-se ... **seleção**!!!

![Seleção](imgs/selecao.png)

Seleção e projeção são termos advindos da *álgebra relacional*, tópico que discutiremos em aulas futuras.

O comando `SELECT` também pode ser usado para efetuar cálculos:

In [62]:
db("SELECT 1 + 1")

Executando query:
(2,)


## Onde foram parar os dados?

MySQL mantém um grande conjunto de variáveis globais para que possamos verificar o estado do sistema. Podemos consultar essas variáveis usando SQL! Afinal, se queremos armazenar informação (neste caso, sobre o próprio sistema), nada melhor que um SGBD!

In [63]:
db("SHOW VARIABLES")

Executando query:
('activate_all_roles_on_login', 'OFF')
('admin_address', '')
('admin_port', '33062')
('admin_ssl_ca', '')
('admin_ssl_capath', '')
('admin_ssl_cert', '')
('admin_ssl_cipher', '')
('admin_ssl_crl', '')
('admin_ssl_crlpath', '')
('admin_ssl_key', '')
('admin_tls_ciphersuites', '')
('admin_tls_version', 'TLSv1.2,TLSv1.3')
('authentication_policy', '*,,')
('auto_generate_certs', 'ON')
('auto_increment_increment', '1')
('auto_increment_offset', '1')
('autocommit', 'OFF')
('automatic_sp_privileges', 'ON')
('avoid_temporal_upgrade', 'OFF')
('back_log', '151')
('basedir', '/usr/')
('big_tables', 'OFF')
('bind_address', '127.0.0.1')
('binlog_cache_size', '32768')
('binlog_checksum', 'CRC32')
('binlog_direct_non_transactional_updates', 'OFF')
('binlog_encryption', 'OFF')
('binlog_error_action', 'ABORT_SERVER')
('binlog_expire_logs_auto_purge', 'ON')
('binlog_expire_logs_seconds', '2592000')
('binlog_format', 'ROW')
('binlog_group_commit_sync_delay', '0')
('binlog_group_commit_s

Para descobrir o valor de uma variável específica, use a cláusula `WHERE`:

In [64]:
db("SHOW VARIABLES WHERE Variable_name = 'version'")

Executando query:
('version', '8.0.45-0ubuntu0.24.04.1')


Para listar todas as variáveis com "dir" no nome:

In [65]:
db("SHOW VARIABLES WHERE Variable_name LIKE '%dir%'")

Executando query:
('basedir', '/usr/')
('binlog_direct_non_transactional_updates', 'OFF')
('character_sets_dir', '/usr/share/mysql/charsets/')
('datadir', '/var/lib/mysql/')
('innodb_data_home_dir', '')
('innodb_directories', '')
('innodb_doublewrite_dir', '')
('innodb_log_group_home_dir', './')
('innodb_max_dirty_pages_pct', '90.000000')
('innodb_max_dirty_pages_pct_lwm', '10.000000')
('innodb_redo_log_archive_dirs', '')
('innodb_temp_tablespaces_dir', './#innodb_temp/')
('innodb_tmpdir', '')
('innodb_undo_directory', './')
('lc_messages_dir', '/usr/share/mysql/')
('plugin_dir', '/usr/lib/mysql/plugin/')
('replica_load_tmpdir', '/tmp')
('slave_load_tmpdir', '/tmp')
('tmpdir', '/tmp')


Observe a variável `datadir`: ela contém o diretório onde o MySQL armazena nossos dados. Por exemplo no Windows: `C:\ProgramData\MySQL\MySQL Server 8.0\Data\`

<center><img src='imgs/datadir.png'/></center>

Note que esse diretório também tem várias chaves criptográficas! Uma vez que o acesso físico ao seu servidor for comprometido, você pode perder completamente o controle, pois o invasor agora tem até mesmo as chaves de acesso!

Dentro do diretório `torneio` temos nossa base de dados no disco:

<center><img src='imgs/torneio.png'/></center>


## Carregando dados em massa

Se você tem um arquivo com vários itens de dados, o melhor jeito de inserí-los no banco de dados é usar os comandos de carga de dados do seu SGBD preferido. No MySQL, veja a documentação do comando `LOAD DATA` em https://dev.mysql.com/doc/refman/8.0/en/load-data.html

Daqui para frente acostume-se a consultar a documentação de todo novo comando que você descobrir!

# `UPDATE`

Vamos alterar informações na nossa base. Para isso vamos usar o comando `UPDATE`. 

Suponha que desejamos alterar o grito de guerra da equipe "Raposas Nerds" de 'sudo vencer!' para 'sudo vencer --force!':

In [66]:
db("UPDATE equipe SET grito='sudo vencer --force!' WHERE nome='Raposas Nerds'")

Executando query:
1 linhas afetadas.


Verificando o resultado:

In [67]:
db("SELECT * from equipe")

Executando query:
('É nois', 'Olha eu mamãe!')
('Raposas Nerds', 'sudo vencer --force!')


**Atividade:** Passe o "Rafael Montaigner" para o time "É nois".

In [68]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(3, 'Luciano Hashimoto', 'Raposas Nerds', 1)
(4, 'Rafael Montaigner', 'Raposas Nerds', 2)
(5, 'Igor Miranda', 'É nois', 2)
(6, 'Andrew Ikeda', 'É nois', 1)
(7, 'Fábio Kurauchi', 'É nois', 0)


In [69]:
# Responda aqui!
db("UPDATE jogador SET nome_equipe='É nois' WHERE nome = 'Rafael Montaigner'")

Executando query:
1 linhas afetadas.


In [70]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(3, 'Luciano Hashimoto', 'Raposas Nerds', 1)
(4, 'Rafael Montaigner', 'É nois', 2)
(5, 'Igor Miranda', 'É nois', 2)
(6, 'Andrew Ikeda', 'É nois', 1)
(7, 'Fábio Kurauchi', 'É nois', 0)


**Atividade:** Tente mudar o nome de "Andrew Ikeda" para "Andrew Gomes da Silva". O que aconteceu?

In [71]:
# Responda aqui!
db("UPDATE jogador SET nome='Andrew Gomes da Silva' WHERE nome='Andrew Ikeda'")

Executando query:
1 linhas afetadas.


In [72]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(3, 'Luciano Hashimoto', 'Raposas Nerds', 1)
(4, 'Rafael Montaigner', 'É nois', 2)
(5, 'Igor Miranda', 'É nois', 2)
(6, 'Andrew Gomes da Silva', 'É nois', 1)
(7, 'Fábio Kurauchi', 'É nois', 0)


**Atividade:** Tente agora mudar o nome do time "É nois" para "Somos nos". O que aconteceu?

In [73]:
# Responda aqui!
db("UPDATE equipe SET nome ='Somos nos' WHERE nome='É nois'")

Executando query:


IntegrityError: 1451 (23000): Cannot delete or update a parent row: a foreign key constraint fails (`torneio`.`jogador`, CONSTRAINT `fk_equipe` FOREIGN KEY (`nome_equipe`) REFERENCES `equipe` (`nome`))

In [74]:
db("SELECT * FROM equipe")

Executando query:
('É nois', 'Olha eu mamãe!')
('Raposas Nerds', 'sudo vencer --force!')


## `COMMIT` e `ROLLBACK`

Note que as ultimas atividades (de `UPDATE`) foram feitas sem executar `connection.commit()`. Neste caso todas as modificações realizadas ainda não foram registradas no banco de dados! Estas mudanças existem apenas na nossa *sessão*. Vamos verificar esse fenômeno. Primeiro vamos ver o estado das nossas tabelas:

In [75]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(3, 'Luciano Hashimoto', 'Raposas Nerds', 1)
(4, 'Rafael Montaigner', 'É nois', 2)
(5, 'Igor Miranda', 'É nois', 2)
(6, 'Andrew Gomes da Silva', 'É nois', 1)
(7, 'Fábio Kurauchi', 'É nois', 0)


In [76]:
db("SELECT * FROM equipe")

Executando query:
('É nois', 'Olha eu mamãe!')
('Raposas Nerds', 'sudo vencer --force!')


Agora vamos verificar o estado do banco de dados lá no MySQL Workbench:

<center><img src='imgs/equipes.png'/></center>

<center><img src='imgs/jogadores.png'/></center>

Está diferente! Isso acontece porque as mudanças da nossa sessão ainda não foram *committed*, não foram registradas em definitivo. Para registrar as mudanças você pode executar o comando `connection.commit()`, ou equivalentemente rodar o comando SQL `COMMIT`. Se você se arrependeu das mudanças realizadas na sessão, jogue elas fora com o comando `connection.rollback()` ou equivalentemente o comando SQL `ROLLBACK`.

Vamos então concretizar nossas mudanças:

In [77]:
connection.commit()
# ou então db("COMMIT"), é a mesma coisa.

Agora vamos verificar lá no MySQL Workbench:

<center><img src='imgs/equipes2.png'/></center>

<center><img src='imgs/jogadores2.png'/></center>

Podemos constatar que as mudanças feitas na nossa sessão agora são visíveis na outra sessão (a do MySQL Workbench). Vamos revisitar futuramente os comandos `COMMIT` e `ROLLBACK` na aula de transações SQL.

## `DELETE`

Vamos remover o 'Fábio Kurauchi' da tabela `jogador`

In [88]:
try:
    db("DELETE FROM jogador WHERE nome='Fábio Kurauchi'")
    connection.commit()
except Exception as e:
    print(f"{type(e)}: {e}")
    connection.rollback()

Executando query:
0 linhas afetadas.


Verificando o resultado:

In [89]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(3, 'Luciano Hashimoto', 'Raposas Nerds', 1)
(4, 'Rafael Montaigner', 'Somos nós', 2)
(5, 'Igor Miranda', 'Somos nós', 2)
(6, 'Andrew Gomes da Silva', 'Somos nós', 1)


**Atividade:** Remova da tabela de jogadores todos aqueles que preferem jogar cowboy.

In [90]:
# Responda aqui!
db("DELETE FROM jogador WHERE preferencia=1")

Executando query:
2 linhas afetadas.


In [93]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(4, 'Rafael Montaigner', 'Somos nós', 2)
(5, 'Igor Miranda', 'Somos nós', 2)


Agora vamos tentar remover o time 'É nois':

In [92]:
try:
    db("DELETE FROM equipe WHERE nome='É nois'")
    connection.commit()
except Exception as e:
    print(f"{type(e)}: {e}")
    connection.rollback()

Executando query:
0 linhas afetadas.


Novamente o malfadado erro `IntegrityError`. Isso acontece porque o SGBD está fazendo exatamente o que você pediu: garantindo a integridade dos dados! Afinal, se removermos a equipe `'É nois'`, o que fazemos com as linhas da tabela-filha que referenciam esse registro?

A restrição de chave estrangeira no nome da equipe (em `script_001.sql` ela foi chamada de `fk_equipe`) especifica, implicitamente, que a ação de apagar um registro deve ser bloqueada se impactar tabelas filhas. Podemos mudar essa restrição.

## `ON UPDATE / ON DELETE`

Vamos reescrever a restrição de chave estrangeira. Copie o script a seguir para `script_002.sql`.

```mysql
USE torneio;

ALTER TABLE jogador
    DROP FOREIGN KEY fk_equipe;

ALTER TABLE jogador
    ADD CONSTRAINT fk_equipe FOREIGN KEY (nome_equipe) REFERENCES equipe (nome)
    ON DELETE CASCADE
    ON UPDATE CASCADE;

```

Feche a conexão atual com o banco de dados: ela está bloqueando alterações no schema. Não se esqueça de efetivar as modificações passadas com `commit()`, se elas não foram *committed* até o momento:

In [94]:
connection.commit()
connection.close()

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Agora rode `script_002.sql` no MySQL Workbench. Se tudo deu certo você deve observar a mudança de ação na restrição de chave estrageira:

</div>

<br>
<center><img src='imgs/cascade.png'/></center>
<br>

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Reabra a conexão e recrie o *helper*:

</div>

In [95]:
connection = mysql.connector.connect(**connection_options)
db = ConnectionHelper(connection)

Vamos agora tentar mudar o nome da equipe 'É nois' para 'Somos nós':

In [96]:
db("UPDATE equipe SET nome='Somos nós' WHERE nome='É nois'")

Executando query:
0 linhas afetadas.


Verificando o resultado:

In [97]:
db("SELECT * from equipe")

Executando query:
('Raposas Nerds', 'sudo vencer --force!')
('Somos nós', 'Olha eu mamãe!')


Vamos verificar o que aconteceu com a tabela `jogador`

In [98]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(4, 'Rafael Montaigner', 'Somos nós', 2)
(5, 'Igor Miranda', 'Somos nós', 2)


Como você pode ver, a mudança de nome foi devidamente propagada, cortesia do `ON UPDATE CASCADE`.

Já o `ON DELETE CASCADE` é mais bruto. Vamos remover o time 'Somos nós':

In [99]:
db("DELETE FROM equipe WHERE nome='Somos nós'")

Executando query:
1 linhas afetadas.


In [100]:
db("SELECT * from equipe")

Executando query:
('Raposas Nerds', 'sudo vencer --force!')


Ok. E os jogadores?

In [101]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)


A ação `CASCADE` associada ao evento `ON DELETE` acabou por limpar a tabela `jogador` tambem.

Vamos cancelar todas essas ações com o comando `rollback()`, que é o oposto de `commit()`

In [102]:
connection.rollback()

Verificando se deu certo:

In [103]:
db("SELECT * FROM jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(4, 'Rafael Montaigner', 'Somos nós', 2)
(5, 'Igor Miranda', 'Somos nós', 2)


**Atividade:** Alem de `CASCADE` e `RESTRICT` existem outros especificadores de ação. Ache a URL da documentação que mostra essas opções.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sua resposta aqui! Dê dois cliques e edite.
    
    https://dev.mysql.com/doc/refman/8.4/en/create-table-foreign-keys.html

    CASCADE: Propaga a ação pelas tabelas filhas (ex: Mudar o nome do time muda as fks nas tabelas filhas)
    SET NULL: Executa a ação na tabela pai e seta o campo de fk na tabela filha como NULL
    RESTRICT/NO ACTION: Rejeita a ação

</div>

## `REPLACE` = `DELETE` + `INSERT`

O comando `REPLACE` lembra um pouco o comando `UPDATE`, e é aí que mora o perigo! `REPLACE` é na verdade uma combinação de `DELETE` seguido de `INSERT`. Vamos exemplificar: suponha que eu quero atualizar o grito do time 'É nois' para 'Vai dar ruim!', mas ao invés de usar o `UPDATE` resolvi usar o `REPLACE`:

In [104]:
db("SELECT * from equipe")
db("SELECT * from jogador")

Executando query:
('Raposas Nerds', 'sudo vencer --force!')
('Somos nós', 'Olha eu mamãe!')
Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(4, 'Rafael Montaigner', 'Somos nós', 2)
(5, 'Igor Miranda', 'Somos nós', 2)


In [105]:
db("REPLACE INTO equipe VALUES ('É nois', 'Vai dar ruim!')")

Executando query:
1 linhas afetadas.


In [106]:
db("SELECT * from equipe")

Executando query:
('É nois', 'Vai dar ruim!')
('Raposas Nerds', 'sudo vencer --force!')
('Somos nós', 'Olha eu mamãe!')


In [107]:
db("SELECT * from jogador")

Executando query:
(2, 'Raul Ayres', 'Raposas Nerds', 0)
(4, 'Rafael Montaigner', 'Somos nós', 2)
(5, 'Igor Miranda', 'Somos nós', 2)


Aparentemente deu ruim!

**Atividade:** Explique

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sua resposta aqui! Dê dois cliques e edite.
    
    O nome da equipe a ser substituida estava incorreto

</div>

Ufa, terminamos a primeira parte! Feche a conexão e siga para a ATV!

In [108]:
connection.rollback()
connection.close()

## Atividade para prática de DML

Faça a atividade para prática de DML. Vamos importar e redefinir o  `ConnectionHelper` para facilitar a vida!

In [1]:
# Importando tudo para facilitar caso precise reiniciar!
import os
import insperautograder.jupyter as ia
import mysql.connector
from mysql.connector import IntegrityError, ProgrammingError
from dotenv import load_dotenv

load_dotenv(override=True)

class ConnectionHelper:

    def __init__(self, connection):
        self.connection = connection

    def __call__(self, query, args=None):
        with self.connection.cursor() as cursor:
            print("Executando query:")
            for result in cursor.execute(query, multi=True):
                if result.with_rows:
                    for row in result.fetchall():
                        print(row)
                else:
                    print(f"{result.rowcount} linhas afetadas.")

### Conferindo a API de Autograding

#### Tarefas e Notas
Vamos conferir as tarefas e notas

In [2]:
ia.tasks()

|    | Atividade    | De                  | Até                 | Conta como ATV?   | % Nota Atraso   |
|---:|:-------------|:--------------------|:--------------------|:------------------|:----------------|
|  0 | newborn      | 2026-02-09 00:00:00 | 2026-05-30 00:00:00 | Não               | 0%              |
|  1 | select01     | 2026-02-11 00:00:00 | 2026-02-22 23:59:59 | Sim               | 25%             |
|  2 | ddl          | 2026-03-02 00:00:00 | 2026-03-09 23:59:59 | Sim               | 25%             |
|  3 | dml          | 2026-03-04 07:00:00 | 2026-03-10 23:59:59 | Sim               | 25%             |
|  4 | agg_join     | 2026-03-09 07:00:00 | 2026-03-15 23:59:59 | Sim               | 25%             |
|  5 | group_having | 2026-03-11 07:00:00 | 2026-03-16 23:59:59 | Sim               | 25%             |

In [3]:
ia.grades(task="dml")

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | dml         | ex01        |      1 |    2.5 |                 0 |               2.5 |
|  1 | dml         | ex02        |      1 |    2.5 |                 0 |               2.5 |
|  2 | dml         | ex03        |      1 |    2.5 |                 0 |               2.5 |
|  3 | dml         | ex04        |      1 |    0   |                 0 |               0   |
|  4 | dml         | ex05        |      1 |    0   |                 0 |               0   |
|  5 | dml         | ex06        |      1 |    0   |                 0 |               0   |

In [4]:
ia.grades(by="TASK")

|    | Tarefa       |   Nota | Conta como ATV?   |
|---:|:-------------|-------:|:------------------|
|  0 | newborn      |  10    | Não               |
|  1 | select01     |   2.5  | Sim               |
|  2 | ddl          |  10    | Sim               |
|  3 | dml          |   1.25 | Sim               |
|  4 | agg_join     |   0    | Sim               |
|  5 | group_having |   0    | Sim               |

Calcular sua média de atividades:


In [5]:
ia.average(excluded_count=2)

|    |   Média de ATV |
|---:|---------------:|
|  0 |           4.58 |

### Criar Base de Dados

Vamos começar pela criação da base de dados. Ela deverá se chamar `vendinha`.

<img src="imgs/vendinha.png">

<p></p>

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Execute no workbench o script `vendinha.sql` disponibilizado pelo professor.

</div>

In [7]:
connection_options_atv = {
    "host": os.getenv("MD_DB_SERVER"),
    "user": os.getenv("MD_DB_USERNAME"),
    "password": os.getenv("MD_DB_PASSWORD"),
    "database": "vendinha"
}

connection_atv = mysql.connector.connect(**connection_options_atv)

db_atv = ConnectionHelper(connection_atv)

In [10]:
connection_atv.commit() #ou connection_atv.rollback() 
connection_atv.close()

**Dica**: Caso necessite executar códigos SQL no Workbench, libere antes a conexão.

```python
connection_atv.commit() #ou connection_atv.rollback() 
connection_atv.close()
```

**Exercício 1**: Faça uma query para inserir a região `"N"` com descrição `"Norte"` na tabela `regiao`.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

**DICA:**
    
Agora que temos a função `db` para conexão com o banco, podemos testar nossas queries aqui mesmo no notebook, alternando comandos de **DML** com `SELECT` para conferir o resultado.
</div>

In [8]:
# Deixa eu ver como está
db_atv("SELECT * FROM regiao")

Executando query:


In [9]:
# Agora deixa eu responder a questão
sql_ex01 = """
    INSERT INTO regiao (regiao, descricao) VALUES ('N', 'Norte');
"""

db_atv(sql_ex01)

Executando query:
1 linhas afetadas.


In [10]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM regiao")

Executando query:
('N', 'Norte')


In [19]:
ia.sender(answer="sql_ex01", task="dml", question="ex01", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex01', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 2**: Em uma única query (sem fazer vários `INSERT`), insira as regiões na tabela `regiao`:

- a região `"NE"` com descrição `"Nordeste"`.
- a região `"SE"` com descrição `"Sudeste"`.
- a região `"S"` com descrição `"Sul"`.
- a região `"CO"` com descrição `"Centro-Oeste"`.

In [11]:
# Deixa eu ver como está
db_atv("SELECT * FROM regiao")

Executando query:
('N', 'Norte')


In [12]:
# Agora deixa eu responder a questão
sql_ex02 = """
    INSERT INTO regiao (regiao, descricao) VALUES ('NE', 'Nordeste'), ('SE', 'Sudeste'), ('S', 'Sul'), ('CO', 'Centro-Oeste');
"""

db_atv(sql_ex02)

Executando query:
4 linhas afetadas.


In [13]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM regiao")

Executando query:
('CO', 'Centro-Oeste')
('N', 'Norte')
('NE', 'Nordeste')
('S', 'Sul')
('SE', 'Sudeste')


In [23]:
ia.sender(answer="sql_ex02", task="dml", question="ex02", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex02', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 3**: Considere que as seguinte querie foi executada:

```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('sp', 'São Paulo', 'SE'),
    ('mg', 'Minas Gerais', 'SE'),
    ('pr', 'Paraná', 'S'),
    ('am', 'Amazonas', 'N'),
    ('ba', 'Bahia', 'NE');
```

**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Crie uma que altere para **maiúsculo** o `uf` apenas dos estados da região **sudeste**.

In [14]:
# Preparando o banco
sql_prep = """INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('sp', 'São Paulo', 'SE'),
    ('mg', 'Minas Gerais', 'SE'),
    ('pr', 'Paraná', 'S'),
    ('am', 'Amazonas', 'N'),
    ('ba', 'Bahia', 'NE');
"""

db_atv(sql_prep)

Executando query:
5 linhas afetadas.


In [15]:
# Deixa eu ver como está
db_atv("SELECT * FROM uf")

Executando query:
('am', 'Amazonas', 'N')
('ba', 'Bahia', 'NE')
('mg', 'Minas Gerais', 'SE')
('pr', 'Paraná', 'S')
('sp', 'São Paulo', 'SE')


In [16]:
# Agora deixa eu responder a questão
sql_ex03 = """
    UPDATE uf SET uf = UPPER(uf) WHERE regiao = 'SE';
"""

db_atv(sql_ex03)

Executando query:
2 linhas afetadas.


In [17]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM uf")

Executando query:
('am', 'Amazonas', 'N')
('ba', 'Bahia', 'NE')
('MG', 'Minas Gerais', 'SE')
('pr', 'Paraná', 'S')
('SP', 'São Paulo', 'SE')


In [29]:
ia.sender(answer="sql_ex03", task="dml", question="ex03", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex03', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 4**: Considere que as seguinte queries foram executadas:

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Perceba que os `uf` estão em maúsculo neste exercício!

</div>


**Tabela `uf`**:
```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');
```

**Tabela `cidade`**:
```mysql
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');
```

**Tabela `vendedor`**:
```mysql
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
```

**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Foi descoberto que todos os vendedores cadastrados em **agosto de 2023** são bots e devem ser removidos da base. Construa sua query!

In [19]:
# Preparando o banco
# Tenta remover todas as referências aos vendedores!
try:
    db_atv("TRUNCATE vendedor_vende_cidade")
except ProgrammingError as e:
    print("Este código foi criado para que consiga executar este exercício novamente\napós ter avançado para os próximos!")
    print(f"Pode ignorar este erro, provavelmente a tabela ainda nao existe!\nErro: {e}")

Executando query:
0 linhas afetadas.


In [20]:
# Preparando o banco
# Tenta remover todas as referências aos vendedores!
try:
    db_atv("TRUNCATE vendedor_ativo_dia")
except ProgrammingError as e:
    print("Este código foi criado para que consiga executar este exercício novamente\napós ter avançado para os próximos!")
    print(f"Pode ignorar este erro, provavelmente a tabela ainda nao existe!\nErro: {e}")

Executando query:
Este código foi criado para que consiga executar este exercício novamente
após ter avançado para os próximos!
Pode ignorar este erro, provavelmente a tabela ainda nao existe!
Erro: 1146 (42S02): Table 'vendinha.vendedor_ativo_dia' doesn't exist


In [42]:
db_atv("DELETE FROM vendedor_vende_cidade")
db_atv("DELETE FROM cidade")
db_atv("DELETE FROM vendedor")
db_atv("DELETE FROM uf")

Executando query:
0 linhas afetadas.
Executando query:
6 linhas afetadas.
Executando query:
7 linhas afetadas.
Executando query:
5 linhas afetadas.


In [43]:
# Preparando o banco
sql_prep = """
-- uf
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');

-- cidade
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');

-- vendedor
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
"""

try:
    db_atv(sql_prep)
except IntegrityError as e:
    print("Ao tentar inserir PKs já existentes, irá falhar!")
    print(f"Provavelmente você já tem os dados!")
    print(f"Erro de integridade: {e}")

Executando query:


In [44]:
# Deixa eu ver como está
db_atv("SELECT * FROM vendedor")

Executando query:
(100, 'Maria Roque', datetime.date(1988, 1, 1), datetime.date(2023, 6, 11), 1)
(101, 'Ana Benedita', datetime.date(1970, 12, 9), datetime.date(2023, 7, 15), 1)
(102, 'Silvio Jardim', datetime.date(1988, 12, 25), datetime.date(2023, 8, 1), 1)
(103, 'Bruna Fontana', datetime.date(1981, 7, 5), datetime.date(2023, 9, 1), 1)
(104, 'Tulio Maravilha', datetime.date(1978, 9, 22), datetime.date(2023, 8, 7), 1)
(105, 'Gino Pereira', datetime.date(1964, 4, 3), datetime.date(2023, 8, 25), 0)
(106, 'Camila Oliveira', datetime.date(1992, 8, 5), datetime.date(2023, 9, 1), 1)
(107, 'Mariana Souza', datetime.date(1985, 8, 29), datetime.date(2023, 9, 1), 1)


In [45]:
# Agora deixa eu responder a questão
sql_ex04 = """
    DELETE FROM vendedor WHERE data_cad BETWEEN '2023-08-01' and '2023-08-31'
"""

db_atv(sql_ex04)

Executando query:
3 linhas afetadas.


In [46]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM vendedor")

Executando query:
(100, 'Maria Roque', datetime.date(1988, 1, 1), datetime.date(2023, 6, 11), 1)
(101, 'Ana Benedita', datetime.date(1970, 12, 9), datetime.date(2023, 7, 15), 1)
(103, 'Bruna Fontana', datetime.date(1981, 7, 5), datetime.date(2023, 9, 1), 1)
(106, 'Camila Oliveira', datetime.date(1992, 8, 5), datetime.date(2023, 9, 1), 1)
(107, 'Mariana Souza', datetime.date(1985, 8, 29), datetime.date(2023, 9, 1), 1)


In [47]:
ia.sender(answer="sql_ex04", task="dml", question="ex04", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex04', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 5**: Considere que as seguinte queries foram executadas:

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Perceba que os `uf` estão em maúsculo neste exercício!

</div>

**Tabela `uf`**:
```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');
```

**Tabela `cidade`**:
```mysql
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');
```

**Tabela `vendedor`**:
```mysql
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
```


**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Dadas as relações de **vendedores** e **cidades onde vendedor vende**:

- Bruna Fontana:
    - São Paulo SP
    - Campinas SP
- Silvio Jardim:
    - São Paulo SP
    - Belo Horizonte MG
    - São Roque de Minas
- Camila Oliveira:
    - Manaus AM
    
Construa uma query para popular a tabela `vendedor_vende_cidade` (Veja Diagrama do Modelo Relacional).

In [48]:
db_atv("DELETE FROM vendedor_vende_cidade")
db_atv("DELETE FROM cidade")
db_atv("DELETE FROM vendedor")
db_atv("DELETE FROM uf")

Executando query:
0 linhas afetadas.
Executando query:
6 linhas afetadas.
Executando query:
5 linhas afetadas.
Executando query:
5 linhas afetadas.


In [49]:
# Preparando o banco
sql_prep = """
-- uf
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');

-- cidade
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');

-- vendedor
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
"""

try:
    db_atv(sql_prep)
except IntegrityError as e:
    print("Ao tentar inserir PKs já existentes, irá falhar!")
    print(f"Provavelmente você já tem os dados!")
    print(f"Erro de integridade: {e}")

Executando query:


In [50]:
# Deixa eu ver como está
db_atv("SELECT * FROM vendedor_vende_cidade")

Executando query:


In [54]:
# Agora deixa eu responder a questão
sql_ex05 = """
INSERT INTO vendedor_vende_cidade (id_vendedor, id_cidade)
VALUES
    (
        (SELECT id FROM vendedor WHERE nome = 'Bruna Fontana'),
        (SELECT id FROM cidade WHERE descricao = 'Campinas' AND uf = 'SP')
    ),
    (
        (SELECT id FROM vendedor WHERE nome = 'Bruna Fontana'),
        (SELECT id FROM cidade WHERE descricao = 'São Paulo' AND uf = 'SP')
    ),
    (
        (SELECT id FROM vendedor WHERE nome = 'Silvio Jardim'),
        (SELECT id FROM cidade WHERE descricao = 'São Paulo' AND uf = 'SP')
    ),
    (
        (SELECT id FROM vendedor WHERE nome = 'Silvio Jardim'),
        (SELECT id FROM cidade WHERE descricao = 'Belo Horizonte' AND uf = 'MG')
    ),
    (
        (SELECT id FROM vendedor WHERE nome = 'Silvio Jardim'),
        (SELECT id FROM cidade WHERE descricao = 'São Roque de Minas' AND uf = 'MG')
    ),
    (
        (SELECT id FROM vendedor WHERE nome = 'Camila Oliveira'),
        (SELECT id FROM cidade WHERE descricao = 'Manaus' AND uf = 'AM')
    );
"""

db_atv(sql_ex05)

Executando query:
6 linhas afetadas.


In [55]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM vendedor_vende_cidade")

Executando query:
(102, 20)
(103, 20)
(103, 21)
(106, 23)
(102, 24)
(102, 25)


In [56]:
ia.sender(answer="sql_ex05", task="dml", question="ex05", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex05', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 6**: Considere que as seguinte queries foram executadas:

**Tabela `uf`**:
```mysql
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');
```

**Tabela `cidade`**:
```mysql
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');
```

**Tabela `vendedor`**:
```mysql
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
```

Ainda, considere uma nova tabela chamada `vendedor_ativo_dia` com o esquema:

| Coluna         | Tipo         | PK (Primary Key?) | Not Null? | Autoinc?
|----------------|--------------|-------------------|-----------|-----------|
| id             | INT          |   True            |    True   |     True  |
| id_vendedor    | INT          |                   |    True   |           |
| ativo          | TYNYINT      |                   |    True   |           |
| data_registro  | DATE         |                   |    True   |           |

Considere que a tabela `vendedor_ativo_dia` é utilizada para criar um registro histórico de todos os vendedores que estão ativos em cada data.

**Obs**: vamos supor que as cinco regiões do BR estão cadastradas. Cadastre-as em sua base local usando a chamada do banco (aqui do notebook mesmo)!

Construa uma query que faça a inserção dos dados na tabela `vendedor_ativo_dia` a partir de um `SELECT` dos vendedores ativos da tabela de vendedor.

<div class="alert alert-danger">

Não deixe a data **hardcoded**. Pesquise como recuperar a data atual do sistema em uma query.

</div>
  
<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Por simplificação, não é necessário criar as constraints de chave extrangeira na tabela `vendedor_ativo_dia`.

</div>

Sugestão de resolução:
- Criar DDL para `vendedor_ativo_dia`
- Criar SELECT para pegar:
    - Vendedores ativos
    - com o dia de hoje
- Inserir estes dados (usando `SELECT`) na tabela `vendedor_ativo_dia`.

Não é necessário enviar a **DDL** para o servidor.

In [61]:
db_atv("DELETE FROM vendedor_vende_cidade")
db_atv("DELETE FROM cidade")
db_atv("DELETE FROM vendedor")
db_atv("DELETE FROM uf")

Executando query:
0 linhas afetadas.
Executando query:
6 linhas afetadas.
Executando query:
8 linhas afetadas.
Executando query:
5 linhas afetadas.


In [62]:
# Preparando o banco
sql_prep = """
-- uf
INSERT INTO vendinha.uf
    (uf, descricao, regiao)
VALUES
    ('SP', 'São Paulo', 'SE'),
    ('MG', 'Minas Gerais', 'SE'),
    ('PR', 'Paraná', 'S'),
    ('AM', 'Amazonas', 'N'),
    ('BA', 'Bahia', 'NE');

-- cidade
INSERT INTO vendinha.cidade
    (id, descricao, uf)
VALUES
    (20, 'São Paulo', 'SP'),
    (21, 'Campinas', 'SP'),
    (22, 'Salvador', 'BA'),
    (23, 'Manaus', 'AM'),
    (24, 'Belo Horizonte', 'MG'),
    (25, 'São Roque de Minas', 'MG');

-- vendedor
INSERT INTO vendinha.vendedor
    (id, nome, data_nasc, data_cad, ativo)
VALUES
    (100, 'Maria Roque', '1988-01-01', '2023-06-11', 1),
    (101, 'Ana Benedita', '1970-12-09', '2023-07-15', 1),
    (102, 'Silvio Jardim', '1988-12-25', '2023-08-01', 1),
    (103, 'Bruna Fontana', '1981-07-05', '2023-09-01', 1),
    (104, 'Tulio Maravilha', '1978-09-22', '2023-08-07', 1),
    (105, 'Gino Pereira', '1964-04-03', '2023-08-25', 0),
    (106, 'Camila Oliveira', '1992-08-05', '2023-09-01', 1),
    (107, 'Mariana Souza', '1985-08-29', '2023-09-01', 1);
"""

try:
    db_atv(sql_prep)
except IntegrityError as e:
    print("Ao tentar inserir PKs já existentes, irá falhar!")
    print(f"Provavelmente você já tem os dados!")
    print(f"Erro de integridade: {e}")

Executando query:


In [63]:
ddl_vendedor_ativo_dia="""
    CREATE TABLE IF NOT EXISTS vendedor_ativo_dia (
        id INTEGER PRIMARY KEY NOT NULL AUTO_INCREMENT,
        id_vendedor INTEGER NOT NULL,
        ativo TINYINT NOT NULL,
        data_registro DATETIME NOT NULL
    );
"""

db_atv(ddl_vendedor_ativo_dia)

Executando query:
0 linhas afetadas.


In [64]:
# Deixa eu ver como está
db_atv("SELECT * FROM vendedor_ativo_dia")

Executando query:


In [65]:
# Agora deixa eu responder a questão (INSERT)
sql_ex06 = """
INSERT INTO vendedor_ativo_dia (id_vendedor, ativo, data_registro)
SELECT id, ativo, CURDATE()
FROM vendedor
WHERE ativo = 1;
"""

db_atv(sql_ex06)

Executando query:
7 linhas afetadas.


In [66]:
# Deixa eu ver o RESULTADO!
db_atv("SELECT * FROM vendedor_ativo_dia")

Executando query:
(1, 100, 1, datetime.datetime(2026, 3, 15, 0, 0))
(2, 101, 1, datetime.datetime(2026, 3, 15, 0, 0))
(3, 102, 1, datetime.datetime(2026, 3, 15, 0, 0))
(4, 103, 1, datetime.datetime(2026, 3, 15, 0, 0))
(5, 104, 1, datetime.datetime(2026, 3, 15, 0, 0))
(6, 106, 1, datetime.datetime(2026, 3, 15, 0, 0))
(7, 107, 1, datetime.datetime(2026, 3, 15, 0, 0))


In [67]:
ia.sender(answer="sql_ex06", task="dml", question="ex06", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex06', style=ButtonStyle()), Output()), _dom_classes=('widget…

### Conferir Notas

Confira se as notas na atividade são as esperadas!

Primeiro na atividade atual!

In [68]:
ia.grades(by="TASK", task="dml")

|    | Tarefa   |   Nota | Conta como ATV?   |
|---:|:---------|-------:|:------------------|
|  0 | dml      |    2.5 | Sim               |

In [69]:
ia.grades(task="dml")

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | dml         | ex01        |      1 |    2.5 |                 0 |               2.5 |
|  1 | dml         | ex02        |      1 |    2.5 |                 0 |               2.5 |
|  2 | dml         | ex03        |      1 |    2.5 |                 0 |               2.5 |
|  3 | dml         | ex04        |      1 |    2.5 |                 0 |               2.5 |
|  4 | dml         | ex05        |      1 |    2.5 |                 0 |               2.5 |
|  5 | dml         | ex06        |      1 |    2.5 |                 0 |               2.5 |

E agora em todas as demais!

In [70]:
ia.grades(by="TASK")

|    | Tarefa       |   Nota | Conta como ATV?   |
|---:|:-------------|-------:|:------------------|
|  0 | newborn      |   10   | Não               |
|  1 | select01     |    2.5 | Sim               |
|  2 | ddl          |   10   | Sim               |
|  3 | dml          |    2.5 | Sim               |
|  4 | agg_join     |    0   | Sim               |
|  5 | group_having |    0   | Sim               |

In [71]:
ia.grades()

|    | Atividade    | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:-------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | newborn      | ex01        |      1 |   10   |                10 |               0   |
|  1 | select01     | ex01        |      1 |    2.5 |                 0 |               2.5 |
|  2 | select01     | ex02        |      1 |    2.5 |                 0 |               2.5 |
|  3 | select01     | ex03        |      1 |    2.5 |                 0 |               2.5 |
|  4 | select01     | ex04        |      1 |    2.5 |                 0 |               2.5 |
|  5 | select01     | ex05        |      1 |    2.5 |                 0 |               2.5 |
|  6 | ddl          | ex02        |      1 |   10   |                10 |               0   |
|  7 | ddl          | ex03        |      1 |   10   |                10 |               0   |
|  8 | ddl          | ex04        |      1 |   10   |                10 |               0   |
|  9 | ddl          | ex05        |      1 |   10   |                10 |               0   |
| 10 | ddl          | ex06        |      1 |   10   |                10 |               0   |
| 11 | ddl          | ex07        |      1 |   10   |                10 |               0   |
| 12 | ddl          | ex09        |      1 |   10   |                10 |               0   |
| 13 | ddl          | ex10        |      1 |   10   |                10 |               0   |
| 14 | ddl          | ex11        |      1 |   10   |                10 |               0   |
| 15 | dml          | ex01        |      1 |    2.5 |                 0 |               2.5 |
| 16 | dml          | ex02        |      1 |    2.5 |                 0 |               2.5 |
| 17 | dml          | ex03        |      1 |    2.5 |                 0 |               2.5 |
| 18 | dml          | ex04        |      1 |    2.5 |                 0 |               2.5 |
| 19 | dml          | ex05        |      1 |    2.5 |                 0 |               2.5 |
| 20 | dml          | ex06        |      1 |    2.5 |                 0 |               2.5 |
| 21 | agg_join     | ex01        |      1 |    0   |                 0 |               0   |
| 22 | agg_join     | ex02        |      1 |    0   |                 0 |               0   |
| 23 | agg_join     | ex03        |      1 |    0   |                 0 |               0   |
| 24 | agg_join     | ex04        |      1 |    0   |                 0 |               0   |
| 25 | agg_join     | ex05        |      1 |    0   |                 0 |               0   |
| 26 | agg_join     | ex06        |      1 |    0   |                 0 |               0   |
| 27 | group_having | ex01        |      1 |    0   |                 0 |               0   |
| 28 | group_having | ex02        |      4 |    0   |                 0 |               0   |
| 29 | group_having | ex03        |      4 |    0   |                 0 |               0   |
| 30 | group_having | ex04        |      4 |    0   |                 0 |               0   |
| 31 | group_having | ex05        |      4 |    0   |                 0 |               0   |
| 32 | group_having | ex06        |      8 |    0   |                 0 |               0   |
| 33 | group_having | ex07        |      6 |    0   |                 0 |               0   |
| 34 | group_having | ex08        |      6 |    0   |                 0 |               0   |
| 35 | group_having | ex09        |     12 |    0   |                 0 |               0   |
| 36 | group_having | ex10        |     10 |    0   |                 0 |               0   |
| 37 | group_having | ex11        |     10 |    0   |                 0 |               0   |
| 38 | group_having | ex12        |      6 |    0   |                 0 |               0   |
| 39 | group_having | ex13        |     12 |    0   |                 0 |               0   |
| 40 | views        | ex01        |      1 |    0   |                 0 |               0   |
| 41 | views        | ex02        |      1 |    0   |                 0 |               0   |
| 42 | views        | ex03        |      1 |    0   |                 0 |               0   |
| 43 | views        | ex04        |      1 |    0   |                 0 |               0   |
| 44 | views        | ex05        |      1 |    0   |                 0 |               0   |
| 45 | views        | ex06        |      1 |    0   |                 0 |               0   |
| 46 | views        | ex07        |      1 |    0   |                 0 |               0   |
| 47 | views        | ex08        |      1 |    0   |                 0 |               0   |
| 48 | views        | ex09        |      1 |    0   |                 0 |               0   |
| 49 | views        | ex10        |      1 |    0   |                 0 |               0   |
| 50 | views        | ex11        |      1 |    0   |                 0 |               0   |
| 51 | views        | ex12        |      1 |    0   |                 0 |               0   |

Calcular sua média de atividades:


In [72]:
ia.average(excluded_count=2)

|    |   Média de ATV |
|---:|---------------:|
|  0 |              5 |